### Setup Environment

In [1]:
!git clone https://github.com/NhuGiap04/Fk-Diffusion-Steering.git

Cloning into 'Fk-Diffusion-Steering'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 351 (delta 92), reused 84 (delta 56), pack-reused 206 (from 3)
Receiving objects: 100% (351/351), 211.01 MiB | 30.86 MiB/s, done.
Resolving deltas: 100% (178/178), done.


In [1]:
%cd Fk-Diffusion-Steering
!git pull

/content/Fk-Diffusion-Steering
Already up to date.


In [2]:
%pip install -r requirements.txt -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Building editable for diffusers (pyproject.toml) ... done


### Evolve Steering

In [3]:
%cd text_to_image

import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ''
# os.environ['HF_HOME'] = ''

import json
import argparse
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from PIL import Image

import torch
from diffusers import DDIMScheduler

/content/Fk-Diffusion-Steering/text_to_image


In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from diffusers import DDIMScheduler
from evolve_diffusers import BaseSDXL

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = BaseSDXL.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/10.3G [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from evolve_diffusers.steer_pipeline import iterative_sample_with_stein


def iterative_stein_loop(
    model,
    prompt,
    *,
    num_loops=4,
    num_particles=4,
    steer_start_timestep=200,
    steer_end_timestep=20,
    base_threshold=0.0,
    stein_step_size=0.04,
    stein_bandwidth=None,
    num_inference_steps=50,
    guidance_scale=5.0,
    guidance_reward_fn="ImageReward",
    max_images_to_show=4,
):
    """Run iterative Stein steering and display loop-level diagnostics + final images."""
    out = iterative_sample_with_stein(
        model=model,
        prompt=prompt,
        num_loops=num_loops,
        num_particles=num_particles,
        steer_start_timestep=steer_start_timestep,
        steer_end_timestep=steer_end_timestep,
        base_threshold=base_threshold,
        stein_step_size=stein_step_size,
        stein_bandwidth=stein_bandwidth,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        guidance_reward_fn=guidance_reward_fn,
    )

    mean_rewards = [float(r.mean().item()) for r in out["rewards"]]
    accepted_counts = [int(a.shape[0]) for a in out["accepted"]]
    thresholds = out["thresholds"]
    loop_idx = np.arange(1, len(mean_rewards) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(loop_idx, mean_rewards, marker="o", label="mean reward")
    axes[0].plot(loop_idx, thresholds, marker="s", label="threshold")
    axes[0].set_xlabel("Loop")
    axes[0].set_ylabel("Score")
    axes[0].set_title("Reward and threshold per loop")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].bar(loop_idx, accepted_counts)
    axes[1].set_xlabel("Loop")
    axes[1].set_ylabel("# accepted particles")
    axes[1].set_title("Accepted particles per loop")
    axes[1].grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()

    final_images = out["results"][-1].images
    n = min(max_images_to_show, len(final_images))
    fig, ax = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        ax = [ax]

    for i in range(n):
        ax[i].imshow(final_images[i])
        ax[i].axis("off")
        ax[i].set_title(f"Final loop sample {i + 1}")

    fig.suptitle("Iterative Stein loop: final generated samples", fontsize=13)
    fig.tight_layout()
    plt.show()

    return out


if "pipe" not in globals():
    raise RuntimeError("Run the steer_sample setup cell first so `pipe` is defined.")

loop_out = iterative_stein_loop(
    model=pipe,
    prompt="a photo of a blue clock and a white cup",
    num_loops=16,
    num_particles=8,
    steer_start_timestep=300,
    steer_end_timestep=100,
    stein_step_size=0,
    guidance_scale=5.0,
)

print("Best mean reward:", loop_out["best_mean_reward"])
print("Num loops returned:", len(loop_out["results"]))

  0%|          | 0/50 [00:00<?, ?it/s]

`metric_to_chase` will be ignored as it only applies to 'LLMGrader' as the `reward_name`


ImageReward.pt:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

load checkpoint from /root/.cache/ImageReward/ImageReward.pt


med_config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

checkpoint loaded
loop=1/16 use_stein=False mean_reward=0.5387 threshold=0.5387 accepted=5 rejected=3


  0%|          | 0/50 [00:00<?, ?it/s]

AssertionError: NaN detected in SVGD vector field